# Interactive Customer Churn Dashboard
**Future Interns — Data Science & Analytics | Task 2**

Interactive Plotly dashboard for customer retention & churn analysis.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('../dataset/Churn_Analysis_Dataset.xlsx')
df['Join_Date'] = pd.to_datetime(df['Join_Date'])
df['Churn_Reason'] = df['Churn_Reason'].replace('NA', 'Retained')
df['Churn_Label'] = df['Churned'].map({0: 'Retained', 1: 'Churned'})
df['Tenure_Group'] = pd.cut(df['Tenure_Months'], bins=[0,12,24,36,60],
                             labels=['0-12 Mo','13-24 Mo','25-36 Mo','37-60 Mo'])
print('Data loaded:', df.shape)

Data loaded: (1000, 16)


## KPI Summary Cards

In [2]:
total = len(df)
churned = df['Churned'].sum()
churn_rate = churned / total * 100
avg_tenure_retained = df[df['Churned']==0]['Tenure_Months'].mean()
avg_tenure_churned = df[df['Churned']==1]['Tenure_Months'].mean()
avg_charge_retained = df[df['Churned']==0]['Monthly_Charge'].mean()

fig = go.Figure()
kpis = [
    ('Total Customers', f'{total:,}', '#3498db', ''),
    ('Churn Rate', f'{churn_rate:.1f}%', '#e74c3c', f'{churned} customers lost'),
    ('Avg Tenure (Active)', f'{avg_tenure_retained:.1f} mo', '#2ecc71', f'vs {avg_tenure_churned:.1f} mo churned'),
    ('Avg Monthly Charge', f'₹{avg_charge_retained:.0f}', '#9b59b6', 'Active customers'),
]

for i, (label, value, color, sub) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode='number',
        value=0,
        number={'valueformat': '', 'font': {'size': 1}},
        title={'text': f'<b style="font-size:22px;color:{color}">{value}</b><br><span style="font-size:13px;color:#666">{label}</span><br><span style="font-size:11px;color:#999">{sub}</span>'},
        domain={'row': 0, 'column': i}
    ))

fig.update_layout(
    grid={'rows': 1, 'columns': 4, 'pattern': 'independent'},
    height=150,
    margin=dict(t=20, b=10, l=10, r=10),
    paper_bgcolor='#f8f9fa',
    title=dict(text='<b>Customer Churn Analysis — KPI Overview</b>', x=0.5, font=dict(size=16))
)
fig.show()

## Churn by Plan & Region

In [3]:
plan_churn = df.groupby('Plan')['Churned'].agg(['sum','count','mean']).reset_index()
plan_churn.columns = ['Plan','Churned','Total','Rate']
plan_churn['Rate_Pct'] = (plan_churn['Rate']*100).round(1)

region_churn = df.groupby('Region')['Churned'].agg(['sum','count','mean']).reset_index()
region_churn.columns = ['Region','Churned','Total','Rate']
region_churn['Rate_Pct'] = (region_churn['Rate']*100).round(1)
region_churn = region_churn.sort_values('Rate_Pct', ascending=False)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Churn Rate by Plan (%)', 'Churn Rate by Region (%)'),
    horizontal_spacing=0.15)

fig.add_trace(go.Bar(
    x=plan_churn['Plan'], y=plan_churn['Rate_Pct'],
    marker_color=['#e74c3c','#9b59b6','#3498db'],
    text=plan_churn['Rate_Pct'].astype(str)+'%',
    textposition='outside', name='Plan'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=region_churn['Region'], y=region_churn['Rate_Pct'],
    marker_color=['#e74c3c','#e67e22','#3498db','#2ecc71'],
    text=region_churn['Rate_Pct'].astype(str)+'%',
    textposition='outside', name='Region'
), row=1, col=2)

fig.update_layout(height=400, showlegend=False,
    title_text='<b>Churn Rate by Plan & Region</b>', title_x=0.5,
    paper_bgcolor='white', plot_bgcolor='#fafafa')
fig.update_yaxes(title_text='Churn Rate (%)', range=[0,28])
fig.show()

## Churn Reasons Sunburst

In [4]:
reasons = df[df['Churned']==1]['Churn_Reason'].value_counts().reset_index()
reasons.columns = ['Reason','Count']

fig = make_subplots(rows=1, cols=2,
    specs=[[{'type':'domain'},{'type':'xy'}]],
    subplot_titles=('Churn Reasons — Share', 'Churn Reasons — Count'))

fig.add_trace(go.Pie(
    labels=reasons['Reason'], values=reasons['Count'],
    hole=0.4, textinfo='label+percent',
    marker_colors=['#e74c3c','#e67e22','#f39c12','#3498db','#9b59b6','#1abc9c']
), row=1, col=1)

fig.add_trace(go.Bar(
    y=reasons['Reason'], x=reasons['Count'],
    orientation='h',
    marker_color=['#e74c3c','#e67e22','#f39c12','#3498db','#9b59b6','#1abc9c'],
    text=reasons['Count'], textposition='outside'
), row=1, col=2)

fig.update_layout(height=400, showlegend=False,
    title_text='<b>Why Are Customers Leaving?</b>', title_x=0.5,
    paper_bgcolor='white')
fig.show()

## Monthly Cohort Churn Trend

In [5]:
cohort = df.groupby('Cohort_Month')['Churned'].agg(['sum','count','mean']).reset_index()
cohort.columns = ['Month','Churned','Total','Rate']
cohort['Rate_Pct'] = (cohort['Rate']*100).round(1)
cohort = cohort[cohort['Month'] != '2024-01']

bar_colors = ['#e74c3c' if v > 15 else '#f39c12' if v > 12 else '#2ecc71'
              for v in cohort['Rate_Pct']]

fig = make_subplots(rows=2, cols=1,
    subplot_titles=('Monthly Cohort Churn Rate (%)', 'Total vs Churned Customers per Cohort'),
    vertical_spacing=0.15)

fig.add_trace(go.Bar(
    x=cohort['Month'], y=cohort['Rate_Pct'],
    marker_color=bar_colors,
    text=cohort['Rate_Pct'].astype(str)+'%',
    textposition='outside', name='Churn Rate'
), row=1, col=1)

fig.add_hline(y=cohort['Rate_Pct'].mean(), line_dash='dash', line_color='navy',
              annotation_text=f'Avg: {cohort["Rate_Pct"].mean():.1f}%', row=1, col=1)

fig.add_trace(go.Bar(x=cohort['Month'], y=cohort['Total'],
    marker_color='#3498db', opacity=0.7, name='Total'), row=2, col=1)
fig.add_trace(go.Bar(x=cohort['Month'], y=cohort['Churned'],
    marker_color='#e74c3c', name='Churned'), row=2, col=1)

fig.update_layout(height=600, barmode='overlay',
    title_text='<b>Monthly Cohort Analysis (2023)</b>', title_x=0.5,
    paper_bgcolor='white', plot_bgcolor='#fafafa',
    legend=dict(orientation='h', y=-0.1))
fig.show()

## Tenure & Monthly Charge vs Churn

In [6]:
tenure_churn = df.groupby('Tenure_Group', observed=True)['Churned'].agg(['sum','count','mean']).reset_index()
tenure_churn.columns = ['Group','Churned','Total','Rate']
tenure_churn['Rate_Pct'] = (tenure_churn['Rate']*100).round(1)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Churn Rate by Tenure Group', 'Monthly Charge vs Tenure (Churned vs Retained)'),
    column_widths=[0.4, 0.6])

fig.add_trace(go.Bar(
    x=tenure_churn['Group'], y=tenure_churn['Rate_Pct'],
    marker_color=['#e74c3c','#e67e22','#2ecc71','#3498db'],
    text=tenure_churn['Rate_Pct'].astype(str)+'%',
    textposition='outside'
), row=1, col=1)

for label, color, sym in [('Retained','#2ecc71','circle'),('Churned','#e74c3c','x')]:
    sub = df[df['Churn_Label']==label]
    fig.add_trace(go.Scatter(
        x=sub['Tenure_Months'], y=sub['Monthly_Charge'],
        mode='markers', name=label,
        marker=dict(color=color, size=5, opacity=0.5, symbol=sym)
    ), row=1, col=2)

fig.update_layout(height=420,
    title_text='<b>Tenure & Charge Analysis</b>', title_x=0.5,
    paper_bgcolor='white', plot_bgcolor='#fafafa',
    legend=dict(orientation='h', y=-0.15))
fig.update_yaxes(title_text='Churn Rate (%)', range=[0,28], row=1, col=1)
fig.update_xaxes(title_text='Tenure (Months)', row=1, col=2)
fig.update_yaxes(title_text='Monthly Charge (₹)', row=1, col=2)
fig.show()

## Payment Method & Support Calls

In [7]:
pay_churn = df.groupby('Payment_Method')['Churned'].agg(['sum','count','mean']).reset_index()
pay_churn.columns = ['Method','Churned','Total','Rate']
pay_churn['Rate_Pct'] = (pay_churn['Rate']*100).round(1)
pay_churn = pay_churn.sort_values('Rate_Pct', ascending=False)

support_churn = df.groupby('Support_Calls')['Churned'].agg(['sum','count','mean']).reset_index()
support_churn.columns = ['Calls','Churned','Total','Rate']
support_churn['Rate_Pct'] = (support_churn['Rate']*100).round(1)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Churn Rate by Payment Method', 'Churn Rate vs Support Calls'))

fig.add_trace(go.Bar(
    y=pay_churn['Method'], x=pay_churn['Rate_Pct'],
    orientation='h',
    marker_color=['#e74c3c','#e67e22','#3498db','#2ecc71'],
    text=pay_churn['Rate_Pct'].astype(str)+'%',
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=support_churn['Calls'], y=support_churn['Rate_Pct'],
    mode='lines+markers',
    line=dict(color='#e74c3c', width=2.5),
    marker=dict(size=8, color='#e74c3c'),
    fill='tozeroy', fillcolor='rgba(231,76,60,0.1)'
), row=1, col=2)

fig.update_layout(height=380, showlegend=False,
    title_text='<b>Payment Method & Support Calls Analysis</b>', title_x=0.5,
    paper_bgcolor='white', plot_bgcolor='#fafafa')
fig.update_xaxes(title_text='Churn Rate (%)', range=[0,22], row=1, col=1)
fig.update_xaxes(title_text='Number of Support Calls', row=1, col=2)
fig.update_yaxes(title_text='Churn Rate (%)', row=1, col=2)
fig.show()
print('\nDashboard complete! All charts rendered.')


Dashboard complete! All charts rendered.
